# Custom Model from Scratch (NumPy only)

Two classifiers built by hand — no `sklearn`/`torch` for the models themselves, only NumPy.

1. **Logistic regression** — a single sigmoid unit trained with batch gradient descent.
2. **A small 2-layer neural network** — one `tanh` hidden layer + a sigmoid output, trained with full backpropagation.

We train both on the **two-moons** dataset, which is *not* linearly separable. Logistic regression can only draw a straight line, so it plateaus; the neural net bends its decision boundary and separates the classes. The gap between them is exactly what the hidden layer + backprop buys you.

`sklearn` appears only to *generate* the data and split it — never inside a model.

In [ ]:
import numpy as np                                    # all model math: arrays, matmul, elementwise ops
import matplotlib.pyplot as plt                       # plotting only
from sklearn.datasets import make_moons               # toy nonlinear dataset generator
from sklearn.model_selection import train_test_split  # split data into train/test

# Global seeds. NOTE: the models/data below are made reproducible by their own
# random_state / seed arguments, so these two lines don't actually drive anything —
# they're just safe defaults for any ad-hoc NumPy randomness you might add later.
rng = np.random.default_rng(42)  # modern NumPy Generator
np.random.seed(42)               # legacy global RNG

## 1. Data

`X` has shape `(n_samples, 2)` and `y` is a 0/1 label. We standardize the features (zero mean, unit variance) using **only the training statistics** — gradient descent converges far faster on scaled inputs, and applying test data with train stats avoids leakage.

In [ ]:
# make_moons draws two interleaving half-circles -> a classic NOT-linearly-separable problem.
# noise adds Gaussian jitter; random_state makes the sample reproducible.
X, y = make_moons(n_samples=1000, noise=0.20, random_state=42)

# Hold out 25% for testing. stratify=y keeps the 0/1 class ratio identical in both splits.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# --- Feature standardization: (x - mean) / std, done column-wise (per feature) ---
# Compute mean/std on TRAIN ONLY, then apply to both splits. Using test stats would leak
# information from the test set into preprocessing. Scaling also speeds up gradient descent.
mu, sigma = X_train.mean(axis=0), X_train.std(axis=0)  # per-feature mean and std, shape (2,)
X_train = (X_train - mu) / sigma
X_test = (X_test - mu) / sigma                          # TEST uses TRAIN's mu/sigma, not its own

# y comes back as shape (n,); reshape to (n, 1) column vectors so it lines up with the
# model's (n, 1) predictions during the elementwise residual (y_hat - y).
y_train = y_train.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

print(f"train: {X_train.shape}, test: {X_test.shape}")

# Scatter the training data, colored by class, to eyeball the two-moon shape.
plt.figure(figsize=(6, 5))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train.ravel(), cmap="coolwarm", s=12, edgecolor="k", linewidth=0.2)
plt.title("Two moons (standardized, train set)")
plt.xlabel("x1"); plt.ylabel("x2")
plt.show()

## Shared helpers

The **sigmoid** squashes any real number into (0, 1) so it reads as a probability:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

We use a numerically stable version (splitting on the sign of `z`) so large-magnitude inputs don't overflow `exp`.

The loss is **binary cross-entropy** (a.k.a. log-loss), averaged over the batch, with a tiny `eps` inside the log to avoid `log(0)`:

$$\mathcal{L} = -\frac{1}{m}\sum_{i=1}^{m}\Big[ y_i \log \hat{y}_i + (1 - y_i)\log(1 - \hat{y}_i) \Big]$$

In [ ]:
def sigmoid(z):
    """Numerically stable logistic sigmoid: maps any real number into (0, 1) so it reads as a probability."""
    z = np.asarray(z, dtype=float)   # accept lists/ints too; force float so integer inputs don't truncate
    out = np.empty_like(z)           # pre-allocate the output array, same shape/dtype as z

    # The naive 1/(1+exp(-z)) overflows exp() for large |z|. So split the inputs by sign and,
    # per element, use the algebraically-equal formula whose exp() argument is <= 0 (always safe).
    positive = z >= 0                # boolean mask: entries where z is non-negative
    negative = ~positive             # the complement: entries where z is negative

    # For z >= 0: -z <= 0, so exp(-z) lands in (0, 1] -> small and safe, never overflows
    out[positive] = 1.0 / (1.0 + np.exp(-z[positive]))

    # For z < 0: exp(z) sees a negative argument -> in (0, 1), safe. This equals 1/(1+exp(-z)),
    # just rearranged to avoid feeding a large positive number into exp().
    exp_z = np.exp(z[negative])
    out[negative] = exp_z / (1.0 + exp_z)

    return out                       # every element is now a probability in (0, 1)

def bce_loss(y_true, y_pred, eps=1e-12):
    """Average penalty for wrong/unconfident probabilities (binary cross-entropy / log-loss)."""
    y_pred = np.clip(y_pred, eps, 1 - eps)   # keep away from 0 and 1 so log() never sees 0 (-> -inf/NaN)

    # Exactly one term is active per sample (the other is multiplied by 0):
    loss_when_1 = y_true * np.log(y_pred)            # active when truth is 1: big penalty if y_pred is low
    loss_when_0 = (1 - y_true) * np.log(1 - y_pred)  # active when truth is 0: big penalty if y_pred is high

    # Sum the two per-sample terms, average over the batch, negate so LOWER loss = BETTER fit.
    return -np.mean(loss_when_1 + loss_when_0)

def accuracy(y_true, y_pred_prob):
    """Fraction of correct yes/no guesses."""
    guesses = (y_pred_prob >= 0.5).astype(int)   # threshold the probability at 0.5 -> hard 0/1 label
    return np.mean(guesses == y_true)            # mean of a boolean array = fraction that match the truth

def plot_decision_boundary(predict_prob, X, y, title):
    """Paint the model's probability across the space; overlay real points."""
    # 1. grid of points a little bigger than the data (pad each axis by 0.5)
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5   # x1 range, padded
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5   # x2 range, padded
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),      # 300x300 mesh covering the plane
                         np.linspace(y_min, y_max, 300))

    # 2. predict a probability at every one of the 300*300 grid points
    grid = np.c_[xx.ravel(), yy.ravel()]         # flatten the mesh into an (90000, 2) list of (x1, x2) points
    probs = predict_prob(grid).reshape(xx.shape) # predict, then fold the flat vector back into grid shape

    # 3. draw: filled probability field, the 0.5 decision line, and the real data on top
    plt.figure(figsize=(6, 5))                                           # own figure (don't reuse a prior cell's axes)
    plt.contourf(xx, yy, probs, levels=25, cmap="coolwarm", alpha=0.7)   # colored probability field
    plt.contour(xx, yy, probs, levels=[0.5], colors="k", linewidths=1.2) # the decision line (where p = 0.5)
    plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap="coolwarm",
                s=12, edgecolor="k", linewidth=0.2)                       # real data points, colored by class
    plt.title(title); plt.xlabel("x1"); plt.ylabel("x2")
    plt.show()

## 2. Logistic regression from scratch

**Model.** For a feature matrix $X \in \mathbb{R}^{m \times n}$, weights $w \in \mathbb{R}^{n \times 1}$, and bias $b$:

$$z = Xw + b, \qquad \hat{y} = \sigma(z)$$

**Gradients.** A convenient property of sigmoid + cross-entropy is that the messy chain rule collapses to a clean residual $(\hat{y} - y)$:

$$\frac{\partial \mathcal{L}}{\partial w} = \frac{1}{m} X^{\top} (\hat{y} - y), \qquad \frac{\partial \mathcal{L}}{\partial b} = \frac{1}{m}\sum (\hat{y} - y)$$

**Update (gradient descent).** Step downhill by the learning rate $\eta$:

$$w \leftarrow w - \eta \frac{\partial \mathcal{L}}{\partial w}, \qquad b \leftarrow b - \eta \frac{\partial \mathcal{L}}{\partial b}$$

In [ ]:
class LogisticRegressionScratch:
    def __init__(self, lr=0.5, n_iters=2000):
        self.lr = lr              # learning rate: step size for each gradient-descent update
        self.n_iters = n_iters    # number of full-batch gradient-descent steps
        self.w = None             # weight vector, shape (n_features, 1); allocated in fit()
        self.b = 0.0              # bias (intercept) scalar
        self.history = []         # loss sampled every 50 iters, for the learning curve

    def fit(self, X, y):
        m, n = X.shape                # m = #samples, n = #features
        self.w = np.zeros((n, 1))     # zero init is fine here (loss is convex for logreg)
        self.b = 0.0
        for i in range(self.n_iters):
            # ---- forward: linear score -> probability ----
            z = X @ self.w + self.b   # (m, 1) raw scores
            y_hat = sigmoid(z)        # (m, 1) predicted probabilities

            # ---- gradients of BCE w.r.t. w and b ----
            error = y_hat - y         # (m, 1) residual; sigmoid+BCE make the gradient this clean
            dw = (X.T @ error) / m    # (n, 1) mean weight gradient
            db = np.mean(error)       # scalar bias gradient

            # ---- gradient-descent step: move parameters downhill ----
            self.w -= self.lr * dw
            self.b -= self.lr * db

            if i % 50 == 0:
                self.history.append(bce_loss(y, y_hat))  # record progress
        return self

    def predict_proba(self, X):
        return sigmoid(X @ self.w + self.b)   # same forward pass, no learning


# Train, then report accuracy on both splits.
logreg = LogisticRegressionScratch(lr=0.5, n_iters=2000).fit(X_train, y_train)

print(f"train acc: {accuracy(y_train, logreg.predict_proba(X_train)):.3f}")
print(f"test  acc: {accuracy(y_test,  logreg.predict_proba(X_test)):.3f}")

# A single linear boundary can't wrap around the moons.
plot_decision_boundary(logreg.predict_proba, X_test, y_test, "Logistic regression — linear boundary")

The boundary is a straight line, so it can't follow the curve between the two moons — accuracy tops out around the high-80s%. That's the ceiling for a linear model here.

## 3. A 2-layer neural network from scratch

Architecture: `2 inputs → H tanh hidden units → 1 sigmoid output`.

**Forward pass**
$$Z_1 = X W_1 + b_1, \quad A_1 = \tanh(Z_1)$$
$$Z_2 = A_1 W_2 + b_2, \quad A_2 = \sigma(Z_2) = \hat{y}$$

**Backpropagation** (chain rule, layer by layer). Using $\frac{d}{dz}\tanh(z) = 1 - \tanh^2(z)$:
$$dZ_2 = A_2 - y$$
$$dW_2 = \tfrac{1}{m} A_1^{\top} dZ_2, \qquad db_2 = \tfrac{1}{m}\textstyle\sum dZ_2$$
$$dZ_1 = (dZ_2 \, W_2^{\top}) \odot (1 - A_1^2)$$
$$dW_1 = \tfrac{1}{m} X^{\top} dZ_1, \qquad db_1 = \tfrac{1}{m}\textstyle\sum dZ_1$$

Then the same gradient-descent update on every parameter. Weights are initialized small and random (not zero) to **break symmetry** — if all hidden units started identical they'd stay identical and the hidden layer would be useless.

In [ ]:
class NeuralNetScratch:
    def __init__(self, n_hidden=8, lr=0.5, n_iters=5000, seed=42):
        self.n_hidden = n_hidden  # number of tanh units in the hidden layer
        self.lr = lr              # learning rate
        self.n_iters = n_iters    # gradient-descent steps
        self.seed = seed          # seed for reproducible weight init
        self.history = []         # loss sampled every 100 iters

    def _init_params(self, n_features):
        rng = np.random.default_rng(self.seed)
        # Random (NOT zero) init breaks symmetry so hidden units learn different features.
        # sqrt(1/fan_in) scaling (Xavier-ish) keeps initial activations from saturating tanh.
        self.W1 = rng.standard_normal((n_features, self.n_hidden)) * np.sqrt(1.0 / n_features)  # (2, H)
        self.b1 = np.zeros((1, self.n_hidden))                                                  # (1, H)
        self.W2 = rng.standard_normal((self.n_hidden, 1)) * np.sqrt(1.0 / self.n_hidden)        # (H, 1)
        self.b2 = np.zeros((1, 1))

    def _forward(self, X):
        Z1 = X @ self.W1 + self.b1   # (m, H) hidden pre-activations
        A1 = np.tanh(Z1)             # (m, H) hidden activations (the nonlinearity -> curved boundary)
        Z2 = A1 @ self.W2 + self.b2  # (m, 1) output pre-activation
        A2 = sigmoid(Z2)             # (m, 1) output probability
        cache = (X, A1, A2)          # stash intermediates that backprop needs
        return A2, cache

    def fit(self, X, y):
        m, n = X.shape
        self._init_params(n)
        for i in range(self.n_iters):
            # ---- forward ----
            A2, (X_, A1, _) = self._forward(X)

            # ---- backward: propagate the error from the output back to each layer ----
            dZ2 = A2 - y                              # (m, 1) same clean residual as logreg
            dW2 = (A1.T @ dZ2) / m                     # (H, 1) grad for output weights
            db2 = np.mean(dZ2, axis=0, keepdims=True)  # (1, 1) grad for output bias

            # backprop through tanh: multiply by its derivative (1 - tanh^2) = (1 - A1**2)
            dZ1 = (dZ2 @ self.W2.T) * (1 - A1 ** 2)    # (m, H)
            dW1 = (X_.T @ dZ1) / m                     # (n, H) grad for hidden weights
            db1 = np.mean(dZ1, axis=0, keepdims=True)  # (1, H) grad for hidden bias

            # ---- gradient-descent update (all four parameter blocks) ----
            self.W2 -= self.lr * dW2
            self.b2 -= self.lr * db2
            self.W1 -= self.lr * dW1
            self.b1 -= self.lr * db1

            if i % 100 == 0:
                self.history.append(bce_loss(y, A2))
        return self

    def predict_proba(self, X):
        A2, _ = self._forward(X)   # only the output is needed for prediction
        return A2


nn = NeuralNetScratch(n_hidden=8, lr=0.5, n_iters=5000).fit(X_train, y_train)

print(f"train acc: {accuracy(y_train, nn.predict_proba(X_train)):.3f}")
print(f"test  acc: {accuracy(y_test,  nn.predict_proba(X_test)):.3f}")

# The hidden layer lets the boundary curve around the moons.
plot_decision_boundary(nn.predict_proba, X_test, y_test, "Neural net — curved boundary")

## 4. Compare

Loss curves and final test accuracy side by side. The neural net drives its loss lower and follows the moon shape, landing in the mid-to-high 90s% where logistic regression stalls.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Left: training-loss curves. Multiply each x by its logging interval (logreg logged every
# 50 iters, nn every 100) so both curves share a true iteration axis.
ax[0].plot(np.arange(len(logreg.history)) * 50, logreg.history, label="logistic regression")
ax[0].plot(np.arange(len(nn.history)) * 100, nn.history, label="neural net")
ax[0].set_title("Training loss (BCE)"); ax[0].set_xlabel("iteration"); ax[0].set_ylabel("loss")
ax[0].legend()

# Right: final test accuracy for each model.
models = ["LogReg", "NeuralNet"]
test_accs = [
    accuracy(y_test, logreg.predict_proba(X_test)),
    accuracy(y_test, nn.predict_proba(X_test)),
]
bars = ax[1].bar(models, test_accs, color=["#6699cc", "#cc6666"])
ax[1].set_ylim(0, 1); ax[1].set_title("Test accuracy")
for bar, acc in zip(bars, test_accs):   # annotate each bar with its value
    ax[1].text(bar.get_x() + bar.get_width() / 2, acc + 0.02, f"{acc:.3f}", ha="center")

plt.tight_layout(); plt.show()

## 5. Gradient check (sanity test for backprop)

Backprop is easy to get subtly wrong. We verify the analytic gradient against a **numerical** one computed by finite differences:

$$\frac{\partial \mathcal{L}}{\partial \theta} \approx \frac{\mathcal{L}(\theta + \epsilon) - \mathcal{L}(\theta - \epsilon)}{2\epsilon}$$

If the relative difference is tiny (~1e-7), the hand-derived gradients are correct.

In [ ]:
def gradient_check(model, X, y, eps=1e-5):
    """Compare backprop's analytic gradient to a numerical estimate for a few W1 entries."""
    # --- analytic gradient for W1 from one backward pass ---
    A2, (X_, A1, _) = model._forward(X)
    m = X.shape[0]
    dZ2 = A2 - y
    dZ1 = (dZ2 @ model.W2.T) * (1 - A1 ** 2)
    dW1_analytic = (X_.T @ dZ1) / m

    # --- numerical gradient via central finite differences, for 5 random W1 entries ---
    rng = np.random.default_rng(0)
    idx = [(int(rng.integers(model.W1.shape[0])), int(rng.integers(model.W1.shape[1]))) for _ in range(5)]
    max_rel = 0.0
    for (i, j) in idx:
        orig = model.W1[i, j]
        model.W1[i, j] = orig + eps                   # nudge this one weight up by eps
        loss_plus = bce_loss(y, model._forward(X)[0])
        model.W1[i, j] = orig - eps                   # nudge it down by eps
        loss_minus = bce_loss(y, model._forward(X)[0])
        model.W1[i, j] = orig                         # restore the original value
        num = (loss_plus - loss_minus) / (2 * eps)    # slope ~= dLoss/dW at this entry
        ana = dW1_analytic[i, j]
        # relative difference; ~1e-7 means analytic and numeric agree -> backprop is correct
        rel = abs(num - ana) / max(1e-12, abs(num) + abs(ana))
        max_rel = max(max_rel, rel)
        print(f"W1[{i},{j}]  analytic={ana:+.6e}  numeric={num:+.6e}  rel_diff={rel:.2e}")
    print(f"\nmax relative difference: {max_rel:.2e}  ->  {'PASS' if max_rel < 1e-5 else 'CHECK'}")


gradient_check(nn, X_train, y_train)